<a href="https://colab.research.google.com/github/M4rck0/Datos_Masivos/blob/main/Tarea_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejecutar Spark en Google Colab

In [ ]:
%%capture
!pip -q uninstall -y dataproc-spark-connect pyspark py4j # Limpieza e instalación
!pip -q install --no-cache-dir pyspark==3.5.1

# Instalar Java
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq


In [ ]:
import os
from pyspark.sql import SparkSession
from datasets import load_dataset
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Variables de entorno
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

!java -version

In [ ]:
spark = (
    SparkSession.builder
    .appName("SparkEnColab")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

print("Version de spark:", spark.version)

## Prueba de funcionamiento

In [ ]:
df_prueba = spark.createDataFrame([(1,"a"), (2,"b"), (3,"c")], ["id","valor"])
df_prueba.show()

In [ ]:
df_prueba.createOrReplaceTempView("t")
spark.sql("SELECT valor, COUNT(*) AS n FROM t GROUP BY valor").show()

# Dataset elegido y justificación

**Dataset elegido:** `HuggingFaceGECLM/REDDIT_comments` (partición: `technology`).

**Descripción:** Conjunto de comentarios de Reddit que incluye campos como:
- `created_utc` (timestamp),
- `author` (usuario),
- `score` (puntuación del comentario),
- `body` (texto del comentario).

**Justificación:** Se eligió porque es un dataset real y grande, ideal para practicar metadatos con operaciones de PySpark como limpieza, filtrado, agregaciones por tiempo y estadísticas descriptivas, además para practicar con PySpark.

# Carga del dataset con PySpark

In [ ]:
# Parámetros
subreddit = "technology" # Subreddit a descargar
filas_maximas = 200_000 # Límite total de filas a descargar
tam_lote = 50000 # Tamaño del lote
lote = [] # Acumular filas hasta llegar al tamaño del lote
n = 0 # Contador total de filas procesadas
parte = 0 # Número de carpeta que se guarda

# Esquema
esquema = StructType([
    StructField("subreddit", StringType(), True),
    StructField("created_utc", LongType(), True),
    StructField("author", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("body", StringType(), True),
])

# Convierte a int, sino regresa none
def to_int(x):
    try:
        if x is None:
            return None
        return int(x)
    except Exception:
        return None

# Cargar dataset en modo streaming
# streaming = true significa que no se descarga todo a ram
# En su lugar, se itera registro por registro
ds = load_dataset(
    "HuggingFaceGECLM/REDDIT_comments",
    split=subreddit,
    streaming=True
)

for fila in ds:
    # Convertimos cada registro a una tupla con el orden del esquema
    lote.append((
        fila.get("subreddit"),
        to_int(fila.get("created_utc")),
        fila.get("author"),
        to_int(fila.get("score")),
        fila.get("body"),
    ))
    n += 1

    # Cada vez que juntamos filas:
    # 1) Creamos un dataframe de spark con esquema fijo
    # 2) Lo escribimos en parquet usando spark
    # 3) Limpiamos el lote y avanzamos el contador de partes
    if n % tam_lote == 0:
        df_lote = spark.createDataFrame(lote, schema=esquema)

        # Spark escribe a carpetas
        ruta_salida = f"/content/reddit_spark/{subreddit}/parte{parte:03d}"
        df_lote.write.mode("overwrite").parquet(ruta_salida)

        lote = []
        parte += 1
        print(f"Guardado: {ruta_salida} | filas acumuladas: {n}")

    # Terminar si llegamos al límite
    if n >= filas_maximas:
        break


In [ ]:
df_completo = (spark.read
          .option("recursiveFileLookup", "true")
          .parquet(f"/content/reddit_spark/{subreddit}")
          .withColumn("subreddit", F.lit(subreddit))
         )

print("Filas leídas:", df_completo.count())

# Comentarios no borrados
(df_completo
 .filter((F.col("body").isNotNull()) & (F.col("body") != "[deleted]"))
 .select("created_utc", "author", "score", "body")
 .show(10, truncate=120))

# PySpark: filtros, estadísticas descriptivas y operaciones aritméticas

In [ ]:
# Preparación: timestamp y fecha (en el dataset aparece null)
df = (df_completo
      .withColumn("subreddit", F.lit(subreddit))
      .withColumn("created_ts", F.to_timestamp(F.from_unixtime("created_utc")))
      .withColumn("date", F.to_date("created_ts"))
     )

# Filtrar comentarios no borrados
df_filtrado = (df
      .filter(F.col("author").isNotNull())
      .filter(F.col("author") != "[deleted]")
      .filter(F.col("body").isNotNull())
      .filter(F.col("body") != "[deleted]")
      .filter(F.col("score").isNotNull())
)

print("Total original:", df.count())
print("Total validos:", df_filtrado.count())

# score >= 10
df_score = df_filtrado.filter(F.col("score") >= 10)
print("Comentarios con score >= 10:", df_score.count())
df_score.select("date", "author", "score", "body").orderBy(F.desc("score")).show(5, truncate=80)

# Estadísticas descriptivas
df_filtrado.select("score").describe().show()

# Estadísticas
estadisticas = (df_filtrado
    .groupBy("date")
    .agg(
        F.count("*").alias("comentarios"),
        F.avg("score").alias("promedio_score"),
        F.max("score").alias("max_score")
    )
    .orderBy("date")
)
estadisticas.show(10)

# Operaciones aritméticas
df_oparit = df_filtrado.withColumn("longitud_texto", F.length("body"))

df_oparit = df_oparit.withColumn(
    "puntaje_x_100_caracteres",
    F.when(F.col("longitud_texto") > 0,
           (F.col("score") / F.col("longitud_texto")) * 100
    ).otherwise(None)
)

df_oparit.select("author", "score", "longitud_texto", "puntaje_x_100_caracteres").show(5, truncate=80)

# Operación entre registros: diferencia del conteo diario vs día anterior (lag)
w = Window.orderBy("date")

dif_diaria = (estadisticas
    .withColumn("comentarios_previos", F.lag("comentarios").over(w))
    .withColumn("dif_vs_dia_anterior", F.col("comentarios") - F.col("comentarios_previos"))
    .withColumn(
        "porc_cambio_vs_dia_anterior",
        F.when(F.col("comentarios_previos").isNull(), None)
         .otherwise((F.col("dif_vs_dia_anterior") / F.col("comentarios_previos")) * 100)
    )
)

dif_diaria.show(10)